### Stage 2

In [1]:
# Download data
print("Starting data download...")
def download_data(save_folder, base_raw_url, files_to_download):
    import os
    import urllib.request

    os.makedirs(save_folder, exist_ok=True)

    for file_name in files_to_download:
        raw_url = base_raw_url + file_name
        save_path = os.path.join(save_folder, file_name)

        # check if file already exists
        if os.path.exists(save_path):
            print(f"{file_name} already exists. Skipping download.")
            continue
        print(f"Downloading {file_name}...")
        try:
            urllib.request.urlretrieve(raw_url, save_path)
            print(f"Successfully saved to: {save_path}\n")
        except Exception as e:
            print(f"Failed to download {file_name}. Error: {e}\n")
base_raw_url = "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/"

files_to_download = [
    "dontpatronizeme_pcl.tsv",
]

save_folder = "Dont_Patronize_Me_Data"

download_data(save_folder, base_raw_url, files_to_download)

# CSVs (FIXED: use raw.githubusercontent.com + URL-encoded space)
base_raw_url = (
    "https://raw.githubusercontent.com/"
    "Perez-AlmendrosC/dontpatronizeme/master/"
    "semeval-2022/practice%20splits/"
)

download_data(save_folder, base_raw_url, [
    "dev_semeval_parids-labels.csv",
    "train_semeval_parids-labels.csv"
])

print("All downloads complete!")

Starting data download...
dontpatronizeme_pcl.tsv already exists. Skipping download.
dev_semeval_parids-labels.csv already exists. Skipping download.
train_semeval_parids-labels.csv already exists. Skipping download.
All downloads complete!


In [2]:
# Process .tsv files

# -- dontpatronizeme_pcl.tsv contains paragraphs annotated with a label from 0 (not containing PCL) to 4 (being highly patronizing or condescending) towards vulnerable communities.
# It contains one instance per line with the following format:

# 	- <par_id> <tab> <art_id> <tab> <keyword> <tab> <country_code> <tab> <text> <tab> <label>

	# where
	# - <par_id> is a unique id for each one of the paragraphs in the corpus.
	# - <art_id> is the document id in the original NOW corpus (News on Web: https://www.english-corpora.org/now/).
	# - <keyword> is the search term used to retrieve texts about a target community.
	# - <country_code> is a two-letter ISO Alpha-2 country code for the source media outlet.
	# - <text> is the paragraph containing the keyword.
	# - <label> is an integer between 0 and 4. Each paragraph has been annotated by two annotators as 0 (No PCL), 1 (borderline PCL) and 2 (contains PCL). The combined annotations have been used in the following graded scale:


import pandas as pd
import os

def load_pcl_data(file_path):
    # Column names based on the structure you provided
    # Note: You mentioned <par_id> <art_id> <keyword> <country_code> <text> <label>
    columns = ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']

    # We use sep='\t' for tab-separated files.
    # 'on_bad_lines' is set to skip to avoid errors if the README headers are still in the file.
    # 'quoting=3' (QUOTE_NONE) is often needed if the text contains unescaped quotes.
    df = pd.read_csv(
        file_path,
        sep='\t',
        skiprows=4, # The DPM dataset usually has 4 rows of info text at the top
        names=columns,
        index_col=False,
        on_bad_lines='skip'
    )

    # Clean up: Drop rows where the label or text is missing
    df = df.dropna(subset=['text', 'label'])

    # Convert label to integer (sometimes read as float)
    df['label'] = df['label'].astype(int)

    return df

# --- Usage ---
pcl_path = os.path.join(save_folder, "dontpatronizeme_pcl.tsv")
df_pcl = load_pcl_data(pcl_path)

print(f"Loaded {len(df_pcl)} rows.")
print(df_pcl[['par_id', 'text', 'label']].head())



Loaded 10468 rows.
   par_id                                               text  label
0       1  We 're living in times of absolute insanity , ...      0
1       2  In Libya today , there are countless number of...      0
2       3  White House press secretary Sean Spicer said t...      0
3       4  Council customers only signs would be displaye...      0
4       5  " Just like we received migrants fleeing El Sa...      0


In [3]:
# Process split files data



import pandas as pd

import os



# Define paths (assuming they are in the folder we created earlier)

train_path = os.path.join("Dont_Patronize_Me_Data", "train_semeval_parids-labels.csv")

dev_path = os.path.join("Dont_Patronize_Me_Data", "dev_semeval_parids-labels.csv")



# Load the CSVs

# Note: If these files don't have a header row, use header=None and names=['par_id', 'label']

df_train_ids = pd.read_csv(train_path)

print(df_train_ids.head()) # Check the structure of the loaded DataFrame

df_dev_ids = pd.read_csv(dev_path)

print(df_dev_ids.head()) # Check the structure of the loaded DataFrame



# Extract par_id column into Python lists

# We use .astype(str) to ensure IDs are strings,

# which helps when matching against the main .tsv file later.

train_par_ids = df_train_ids['par_id'].astype(str).tolist()

dev_par_ids = df_dev_ids['par_id'].astype(str).tolist()



# Verification

print(f"Extracted {len(train_par_ids)} IDs for Training.")

print(f"Extracted {len(dev_par_ids)} IDs for Development/Validation.")

print(f"First 5 Train IDs: {train_par_ids[:5]}")

   par_id                  label
0    4341  [1, 0, 0, 1, 0, 0, 0]
1    4136  [0, 1, 0, 0, 0, 0, 0]
2   10352  [1, 0, 0, 0, 0, 1, 0]
3    8279  [0, 0, 0, 1, 0, 0, 0]
4    1164  [1, 0, 0, 1, 1, 1, 0]
   par_id                  label
0    4046  [1, 0, 0, 1, 0, 0, 0]
1    1279  [0, 1, 0, 0, 0, 0, 0]
2    8330  [0, 0, 1, 0, 0, 0, 0]
3    4063  [1, 0, 0, 1, 1, 1, 0]
4    4089  [1, 0, 0, 0, 0, 0, 0]
Extracted 8375 IDs for Training.
Extracted 2094 IDs for Development/Validation.
First 5 Train IDs: ['4341', '4136', '10352', '8279', '1164']


In [ ]:
# Install necessary libraries
!pip install -q transformers[torch] datasets nlpaug sacremoses sentencepiece scikit-learn tqdm

import os
import urllib.request
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# 1. Download data
print("Starting data download...")
def download_data(save_folder, base_raw_url, files_to_download):
    os.makedirs(save_folder, exist_ok=True)
    for file_name in files_to_download:
        raw_url = base_raw_url + file_name
        save_path = os.path.join(save_folder, file_name)
        if os.path.exists(save_path):
            print(f"{file_name} already exists. Skipping download.")
            continue
        print(f"Downloading {file_name}...")
        urllib.request.urlretrieve(raw_url, save_path)

save_folder = "Dont_Patronize_Me_Data"
# Main TSV
download_data(save_folder, "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/", ["dontpatronizeme_pcl.tsv"])
# Splits
download_data(save_folder, "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/", ["dev_semeval_parids-labels.csv", "train_semeval_parids-labels.csv"])
print("All downloads complete!")

In [ ]:
# 2. Process TSV and Merge with Splits
def load_pcl_data(file_path):
    columns = ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']
    df = pd.read_csv(file_path, sep='\t', skiprows=4, names=columns, index_col=False, on_bad_lines='skip')
    df = df.dropna(subset=['text', 'label'])
    # Binary Labeling: 0,1 -> 0 (No PCL) | 2,3,4 -> 1 (PCL)
    df['label_binary'] = df['label'].apply(lambda x: 1 if x >= 2 else 0)
    df['par_id'] = df['par_id'].astype(str)
    return df

pcl_path = os.path.join(save_folder, "dontpatronizeme_pcl.tsv")
df_all = load_pcl_data(pcl_path)

# Load split IDs
train_ids = pd.read_csv(os.path.join(save_folder, "train_semeval_parids-labels.csv"))['par_id'].astype(str).tolist()
dev_ids = pd.read_csv(os.path.join(save_folder, "dev_semeval_parids-labels.csv"))['par_id'].astype(str).tolist()

# Create split DataFrames
train_set = df_all[df_all['par_id'].isin(train_ids)].copy()
dev_set = df_all[df_all['par_id'].isin(dev_ids)].copy()

print(f"Training set: {len(train_set)} | Dev set: {len(dev_set)}")

In [ ]:
import nlpaug.augmenter.word as naw

# 3. Stratified Split and Augmentation
new_train_set, test_set = train_test_split(
    train_set, test_size=0.15, random_state=42, stratify=train_set['label_binary']
)

print("Starting Back-Translation (PCL class only)...")
pcl_subset = new_train_set[new_train_set['label_binary'] == 1]['text'].tolist()

# Initialize Augmenters
back_translation_aug = naw.BackTranslationAug(
    from_model_name='facebook/wmt19-en-de', 
    to_model_name='facebook/wmt19-de-en',
    device='cuda'
)
random_del_aug = naw.RandomWordAug(action="delete", aug_p=0.15)

# Generate synthetic PCL data
bt_texts = []
batch_size = 16
for i in tqdm(range(0, len(pcl_subset), batch_size), desc="Augmenting"):
    batch = pcl_subset[i:i+batch_size]
    bt_texts.extend(back_translation_aug.augment(batch))

rd_texts = random_del_aug.augment(pcl_subset)

# Merge back
augmented_df = pd.concat([
    new_train_set,
    pd.DataFrame({'text': bt_texts, 'label_binary': 1}),
    pd.DataFrame({'text': rd_texts, 'label_binary': 1})
], ignore_index=True).sample(frac=1.0, random_state=42)

print(f"New Training Size: {len(augmented_df)}")

In [ ]:
# 4. DeBERTa Tokenization
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=256)

train_ds = Dataset.from_pandas(augmented_df[['text', 'label_binary']]).rename_column("label_binary", "labels")
dev_ds = Dataset.from_pandas(dev_set[['text', 'label_binary']]).rename_column("label_binary", "labels")
test_ds = Dataset.from_pandas(test_set[['text', 'label_binary']]).rename_column("label_binary", "labels")

tokenized_train = train_ds.map(tokenize_fn, batched=True)
tokenized_dev = dev_ds.map(tokenize_fn, batched=True)
tokenized_test = test_ds.map(tokenize_fn, batched=True)

tokenized_train.set_format("torch")
tokenized_dev.set_format("torch")
tokenized_test.set_format("torch")

In [ ]:
# 5. Define Weighted Loss and Metrics
neg = len(augmented_df[augmented_df['label_binary'] == 0])
pos = len(augmented_df[augmented_df['label_binary'] == 1])
weights = torch.tensor([1.0, neg/pos], dtype=torch.float).to("cuda")

class DebertaWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple): logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, pos_label=1)}

In [ ]:
# 6. Training
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir="./results_deberta",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False # Stability fix for DeBERTa-v3
)

trainer = DebertaWeightedTrainer(
    model=model, args=args, 
    train_dataset=tokenized_train, 
    eval_dataset=tokenized_dev, 
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# 7. Final Predictions for dev.txt
print("Generating dev.txt predictions...")
output = trainer.predict(tokenized_dev)
final_preds = np.argmax(output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions, axis=-1)

with open("dev.txt", "w") as f:
    for p in final_preds:
        f.write(f"{p}\n")

print("✅ dev.txt saved and ready for GitHub.")